In [1]:
import torch
import numpy as np
import pandas as pd
import gpytorch
from matplotlib import pyplot as plt
import scipy.io as sio
import os
from Modules import ExactGPModel
from 

%matplotlib inline
%load_ext autoreload
%autoreload 2


In [2]:
########################################## load the data ############
i = 0
data = sio.loadmat('data/Delay/all_'+str(i)+'.mat')

ALT = data['out'][:,0]
mLat = data['out'][:,1]
mLon = data['out'][:,2]
mLT = data['out'][:,3]
VTEC0 = data['out'][:,4]

varis = data['out'][:,5:14]
DoY = data['out'][:,14]
VTEC1 = data['out'][:,15]
NmF2 = data['out'][:,16]

ind = np.where(np.abs(mLat>60) \
                   & (~np.isnan(varis[:,0])) \
                   & (~np.isnan(varis[:,1])) \
                   & (~np.isnan(varis[:,2])) \
                   & (~np.isnan(varis[:,3])) \
                   & (~np.isnan(varis[:,4])) \
                   & (~np.isnan(varis[:,5])) \
                   & (~np.isnan(varis[:,6])) \
                   & (~np.isnan(varis[:,7])) \
                   & (~np.isnan(varis[:,8])) \
                   #& (VTEC1 > 0))[0]
                  )[0]

def seed_torch(seed):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_torch(1029)

X = torch.from_numpy(data['out'][ind,1:15]).type( torch.FloatTensor )
Y = torch.from_numpy(data['out'][ind,16]).type( torch.FloatTensor )

likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ExactGPModel(X, Y, likelihood)

print(X.shape)
print(Y.shape)

X = X / 

torch.Size([158733, 14])
torch.Size([158733])


In [3]:
import ipdb
# this is for running the notebook in our testing framework
smoke_test = ('CI' in os.environ)
training_iter = 2 if smoke_test else 50


# Find optimal model hyperparameters
model.train()
likelihood.train()

# Use the adam optimizer
optimizer = torch.optim.Adam([
    {'params': model.parameters()},  # Includes GaussianLikelihood parameters
], lr=0.1)

# "Loss" for GPs - the marginal log likelihood
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(training_iter):
    # Zero gradients from previous iteration
    optimizer.zero_grad()
    # Output from model
    output = model(X)
    # Calc loss and backprop gradients
    ipdb.set_trace()
    loss = -mll(output, Y)
    loss.backward()
    print('Iter %d/%d - Loss: %.3f   lengthscale: %.3f   noise: %.3f' % (
        i + 1, training_iter, loss.item(),
        model.covar_module.base_kernel.lengthscale.item(),
        model.likelihood.noise.item()
    ))
    optimizer.step()

> <ipython-input-3-61d5bf949e79>(26)<module>()
     25     ipdb.set_trace()
---> 26     loss = -mll(output, Y)
     27     loss.backward()

ipdb> n
RuntimeError: [enforce fail at CPUAllocator.cpp:64] . DefaultCPUAllocator: can't allocate memory: you tried to allocate 100784661156 bytes. Error code 12 (Cannot allocate memory)
> <ipython-input-3-61d5bf949e79>(26)<module>()
     25     ipdb.set_trace()
---> 26     loss = -mll(output, Y)
     27     loss.backward()

ipdb> n
--Return--
None
> <ipython-input-3-61d5bf949e79>(26)<module>()
     25     ipdb.set_trace()
---> 26     loss = -mll(output, Y)
     27     loss.backward()

ipdb> n
RuntimeError: [enforce fail at CPUAllocator.cpp:64] . DefaultCPUAllocator: can't allocate memory: you tried to allocate 100784661156 bytes. Error code 12 (Cannot allocate memory)
> /export/scratch2/andong/Workspace/Python/test/lib/python3.7/site-packages/IPython/core/interactiveshell.py(3326)run_code()
   3325                 else:
-> 3326                    

BdbQuit: 